In [1]:
import numpy as np
import pandas as pd
import sys
import time
import numpy as np
import torch
import torch.nn.functional as F

from pathlib import Path
from torch.utils.data import DataLoader
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from scipy.stats import spearmanr

ROOT = Path.cwd().resolve().parent   # /mnt/primary/Main
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from loader.load import load_aligned_plans_and_runs

from uncertainty_prediction.src import *
from uncertainty_prediction.new_config import *

from uncertainty_prediction.baselines.deterministic.mscn.data import get_train_test_datasets_from_trino
from uncertainty_prediction.baselines.deterministic.mscn.model import SetConv
from uncertainty_prediction.baselines.deterministic.mscn.util import unnormalize_labels

In [2]:

"""
Load query plan and runs
"""
# any set of plans is fine here (... assuming that plans are identical across runs)
queries_dir = "/mnt/lakehouse-raw-results/tpcds/lakehouse-a/20260429-135222Z/queries"

plans_by_query, runs_by_query, common = load_aligned_plans_and_runs(
    queries_dir=queries_dir,
    run_ids=RUN_IDS,
    collection=COLLECTION_NAME,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    metric=METRIC,
    xcol=XCOL,
    ycol=YCOL,
    parsed_results_root=PARSED_RESULTS_ROOT,
    canon_fn=canon_qid,
    min_runs=1,
    min_points_per_run=2,
    require_cols=(XCOL, YCOL),
)

In [3]:

"""
Load lakehouse profiles
"""

plans_by_query, profiles_by_query, common = load_aligned_plans_and_profiles(
    queries_dir=queries_dir,
    run_ids=RUN_IDS,
    collection=COLLECTION_NAME,
    schema=SCHEMA_NAME,
    instance=LAKEHOUSE_INSTANCE_NAME,
    parsed_results_root=PARSED_RESULTS_ROOT,
    canon_fn=canon_qid,
    profile_filename="node_profiles.csv",
    profile_types=["before"],
    min_profiles_per_query=1,
)

In [4]:

"""
Split into training and testing sets
"""

train_qids, test_qids = split_query_ids(
    common,
    seed=SEED,
    test_frac=TEST_FRAC,
)
print("n_train:", len(train_qids), "n_test:", len(test_qids))

runs_train, runs_test = make_train_test_run_split(
    runs_by_query,
    train_qids=train_qids,
    test_qids=test_qids,
)

n_train: 351 n_test: 87


In [7]:

"""
Generate training and testing sets 
"""

(
    dicts,
    label_stats,
    labels_train_norm,
    labels_test_norm,
    max_num_samples,
    max_num_predicates,
    max_num_joins,
    train_dataset,
    test_dataset,
    kept_train_qids,
    kept_test_qids,
) = get_train_test_datasets_from_trino(
    plans_by_query=plans_by_query,
    runs_by_query=runs_by_query,
    train_qids=train_qids,
    test_qids=test_qids,
    xcol=XCOL,
    runtime_mode="mean",
)


Number of training samples: 351
Number of validation samples: 87
Created TensorDataset for training data
Created TensorDataset for validation data


In [8]:
def qerror_numpy(y_pred, y_true, eps=1e-12):
    y_pred = np.asarray(y_pred, dtype=float).reshape(-1)
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    return np.maximum((y_pred + eps) / (y_true + eps), (y_true + eps) / (y_pred + eps))


def evaluate_runtime_metrics(preds_unnorm, labels_unnorm):
    preds_unnorm = np.asarray(preds_unnorm, dtype=float).reshape(-1)
    labels_unnorm = np.asarray(labels_unnorm, dtype=float).reshape(-1)

    qerr = qerror_numpy(preds_unnorm, labels_unnorm)

    metrics = {
        "mae": float(mean_absolute_error(labels_unnorm, preds_unnorm)),
        "rmse": float(np.sqrt(mean_squared_error(labels_unnorm, preds_unnorm))),
        "r2": float(r2_score(labels_unnorm, preds_unnorm)),
        "median_q_error": float(np.median(qerr)),
        "p95_q_error": float(np.percentile(qerr, 95)),
        "mean_q_error": float(np.mean(qerr)),
        "gmean_q_error": float(np.exp(np.mean(np.log(qerr + 1e-12)))),
    }
    return metrics


def run_epoch(model, loader, optimizer=None, device="cpu"):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    n_batches = 0

    preds = []
    targets = []

    t_total = 0.0

    for batch in loader:
        samples, predicates, joins, y, sample_masks, predicate_masks, join_masks = batch

        samples = samples.to(device)
        predicates = predicates.to(device)
        joins = joins.to(device)
        y = y.to(device)
        sample_masks = sample_masks.to(device)
        predicate_masks = predicate_masks.to(device)
        join_masks = join_masks.to(device)

        if training:
            optimizer.zero_grad()

        t0 = time.time()
        out = model(samples, predicates, joins, sample_masks, predicate_masks, join_masks)
        t_total += time.time() - t0

        loss = F.mse_loss(out, y)

        if training:
            loss.backward()
            optimizer.step()

        total_loss += loss.item()
        n_batches += 1

        preds.append(out.detach().cpu().numpy())
        targets.append(y.detach().cpu().numpy())

    preds = np.vstack(preds).reshape(-1)
    targets = np.vstack(targets).reshape(-1)

    return {
        "loss": total_loss / max(n_batches, 1),
        "preds_norm": preds,
        "targets_norm": targets,
        "pred_time_s": t_total,
    }


# -------- build loaders --------
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

# -------- infer feature dims from one batch --------
batch = next(iter(train_loader))
sample_feats = batch[0].shape[-1]
predicate_feats = batch[1].shape[-1]
join_feats = batch[2].shape[-1]

print("sample_feats:", sample_feats)
print("predicate_feats:", predicate_feats)
print("join_feats:", join_feats)

# -------- model --------
device = "cuda" if torch.cuda.is_available() else "cpu"
hid_units = 128

model = SetConv(
    sample_feats=sample_feats,
    predicate_feats=predicate_feats,
    join_feats=join_feats,
    hid_units=hid_units,
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)

# -------- train --------
num_epochs = 30
best_state = None
best_val_loss = float("inf")

for epoch in range(num_epochs):
    train_out = run_epoch(model, train_loader, optimizer=optimizer, device=device)
    val_out = run_epoch(model, test_loader, optimizer=None, device=device)

    print(
        f"Epoch {epoch+1:03d} | "
        f"train_loss={train_out['loss']:.6f} | "
        f"val_loss={val_out['loss']:.6f}"
    )

    if val_out["loss"] < best_val_loss:
        best_val_loss = val_out["loss"]
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

# -------- restore best --------
if best_state is not None:
    model.load_state_dict(best_state)

# -------- final eval --------
train_eval = run_epoch(model, train_loader, optimizer=None, device=device)
test_eval = run_epoch(model, test_loader, optimizer=None, device=device)

min_val, max_val = label_stats

preds_train_unnorm = unnormalize_labels(train_eval["preds_norm"], min_val, max_val)
labels_train_unnorm = unnormalize_labels(train_eval["targets_norm"], min_val, max_val)

preds_test_unnorm = unnormalize_labels(test_eval["preds_norm"], min_val, max_val)
labels_test_unnorm = unnormalize_labels(test_eval["targets_norm"], min_val, max_val)

train_metrics = evaluate_runtime_metrics(preds_train_unnorm, labels_train_unnorm)
test_metrics = evaluate_runtime_metrics(preds_test_unnorm, labels_test_unnorm)

print("\nTrain metrics")
print(train_metrics)

print("\nTest metrics")
print(test_metrics)

print("\nPrediction time per test sample (ms):",
      1000.0 * test_eval["pred_time_s"] / len(test_dataset))

sample_feats: 20
predicate_feats: 25
join_feats: 38
Epoch 001 | train_loss=0.090379 | val_loss=0.068095
Epoch 002 | train_loss=0.069544 | val_loss=0.067474
Epoch 003 | train_loss=0.065923 | val_loss=0.059135
Epoch 004 | train_loss=0.058367 | val_loss=0.054516
Epoch 005 | train_loss=0.054895 | val_loss=0.049072
Epoch 006 | train_loss=0.052007 | val_loss=0.045240
Epoch 007 | train_loss=0.046300 | val_loss=0.042975
Epoch 008 | train_loss=0.044394 | val_loss=0.038469
Epoch 009 | train_loss=0.041551 | val_loss=0.036359
Epoch 010 | train_loss=0.038678 | val_loss=0.033088
Epoch 011 | train_loss=0.036370 | val_loss=0.032883
Epoch 012 | train_loss=0.037517 | val_loss=0.030088
Epoch 013 | train_loss=0.034326 | val_loss=0.029058
Epoch 014 | train_loss=0.033847 | val_loss=0.028551
Epoch 015 | train_loss=0.035058 | val_loss=0.027423
Epoch 016 | train_loss=0.034061 | val_loss=0.027204
Epoch 017 | train_loss=0.033475 | val_loss=0.028554
Epoch 018 | train_loss=0.034165 | val_loss=0.026259
Epoch 019 | 

In [11]:

def print_metrics_for_excel(metrics):
    print("\t".join(str(v) for v in metrics.values()))

def print_metric_headers_for_excel(metrics):
    print("\t".join(metrics.keys()))
    
def evaluate_runtime_metrics(preds_unnorm, labels_unnorm, eps=1e-12):
    preds_unnorm = np.asarray(preds_unnorm, dtype=float).reshape(-1)
    labels_unnorm = np.asarray(labels_unnorm, dtype=float).reshape(-1)

    if preds_unnorm.shape[0] != labels_unnorm.shape[0]:
        raise ValueError("preds_unnorm and labels_unnorm must have the same length")

    qerr = qerror_numpy(preds_unnorm, labels_unnorm)

    spearman = spearmanr(labels_unnorm, preds_unnorm).statistic
    if spearman is None or np.isnan(spearman):
        spearman = 0.0

    metrics = {
        "mae": float(mean_absolute_error(labels_unnorm, preds_unnorm)),
        "rmse": float(np.sqrt(mean_squared_error(labels_unnorm, preds_unnorm))),
        "spearman": float(spearman),
        "mean_q_error": float(np.mean(qerr)),
        "median_q_error": float(np.median(qerr)),
        "p90_q_error": float(np.percentile(qerr, 90)),
        "p95_q_error": float(np.percentile(qerr, 95)),
    }
    return metrics

test_metrics = evaluate_runtime_metrics(preds_test_unnorm, labels_test_unnorm)

print_metric_headers_for_excel(test_metrics)
print_metrics_for_excel(test_metrics)

mae	rmse	spearman	mean_q_error	median_q_error	p90_q_error	p95_q_error
63.48812290649305	264.6256280316047	0.7566887803455565	5.300728459754165	1.9492952333109639	8.250035479337846	18.48968710091595
